In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['DRIVE'] = str(DRIVE)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')

def git(*args):
    # The token (Colab secret GH_TOKEN: fine-grained, read-only on anitksahu/s1-pii) is passed
    # through GIT_ASKPASS, never on the command line, and errors are printed redacted.
    try:
        tok = (userdata.get('GH_TOKEN') or '').strip()
    except Exception:
        tok = ''
    if not tok:   # public repo: no token needed
        r = subprocess.run(['git', *args], capture_output=True, text=True)
        if r.returncode: raise RuntimeError('git ' + args[0] + ' failed: ' + (r.stderr or '').replace('https://x-access-token@github.com/anitksahu/s1-pii.git', '***')[-800:])
        return r.stdout
    askpass = pathlib.Path('/tmp/askpass.sh'); askpass.write_text('#!/bin/sh\necho "$S1PII_GIT_TOKEN"\n'); askpass.chmod(0o700)
    env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'S1PII_GIT_TOKEN': tok, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    if r.returncode:
        safe = (r.stderr or '').replace(tok, '***').replace('https://x-access-token@github.com/anitksahu/s1-pii.git', '***')
        raise RuntimeError('git ' + args[0] + ' failed: ' + safe[-800:])
    return r.stdout

if not REPO.exists():
    git('clone', 'https://x-access-token@github.com/anitksahu/s1-pii.git', str(REPO))
git('-C', str(REPO), 'fetch')
git('-C', str(REPO), 'switch', 's1d')
git('-C', str(REPO), 'pull', '--ff-only')
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import shlex, sys

def run_visible(args, *, env=None):
    print('$', shlex.join(args), flush=True)
    proc = subprocess.Popen(
        args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    returncode = proc.wait()
    if returncode:
        raise RuntimeError(f'Command failed with exit status {returncode}; full output is above')

run_visible([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchvision', 'torchaudio', 'torchtext', 'torchao'])
run_visible([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'envs/requirements-s1d.txt', '-e', '.[data,model]', 'faker==40.40.0'])
cpu_env = {**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
run_visible([sys.executable, '-c', 'from peft.import_utils import is_torchao_available; is_torchao_available(); from transformers import BertModel, Qwen3Model; print(\"model imports OK\")'], env=cpu_env)
run_visible([sys.executable, '-m', 'pytest', '-q', 'tests/test_s1d.py'], env=cpu_env)

In [ ]:
import json, time, yaml
stage = 'stage0'
stage_root = DRIVE / 's1d'
config = yaml.safe_load(pathlib.Path('s1pii/configs/s1d.yaml').read_text())
cap = float(config['stages'][stage]['cap_a100_hours'])
hours_path = stage_root / 'gpu_hours.jsonl'
hour_rows = [json.loads(line) for line in hours_path.read_text().splitlines() if line.strip()] if hours_path.exists() else []
used = sum(float(row.get('hours', 0)) for row in hour_rows if row.get('stage', stage) == stage)
if used >= cap:
    raise RuntimeError(f'{stage} cannot launch: Drive records {used:.3f}/{cap:g} A100-hours. The hard cap has been reached. Do not run Cell 4.')
control = stage_root / 'CONTROL'
if control.exists() and control.read_text().strip().upper() == 'STOP':
    raise RuntimeError(f'{control} still contains STOP; remove it explicitly before launching')
cuda_probe = (
    'import torch; '
    'assert torch.cuda.is_available(), \"CUDA is unavailable. In Colab select Runtime > Change runtime type > GPU, reconnect, and rerun Cells 1-3.\"; '
    'print(\"CUDA ready:\", torch.cuda.get_device_name(0))'
)
run_visible([sys.executable, '-c', cuda_probe])
stage_root.mkdir(parents=True, exist_ok=True)
(stage_root / 'STATUS').write_text(f"STARTING {stage} {time.strftime('%FT%TZ', time.gmtime())}")
log_dir = DRIVE / 's1d' / 'logs'; log_dir.mkdir(parents=True, exist_ok=True)
log = open(log_dir / f'{stage}-launcher.log', 'a')
proc = subprocess.Popen(['bash', 'scripts/s1d_chain.sh', stage], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('started', proc.pid, 'log', log.name)

In [ ]:
%run scripts/s1d_monitor.py